# 07 · 데이터 품질과 설비 이상 분석

품질 점검은 결측·중복·범위·시간 간격을 확인하는 단계입니다. Z-score와 변화 탐지는 그 다음에 적용합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 품질 요약표

**코드 → 코드 개념**: 품질 문제는 비율과 건수로 함께 기록해야 비교할 수 있다.

**코드 사용법**: 간단한 센서 표에서 누락·중복·범위 위반을 센다.

In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame({"machine": ["A", "A", "A", "B", "B"],
                   "cycle": [1, 2, 2, 1, 2],
                   "vibration": [2.1, np.nan, np.nan, 2.5, 12.0]})
report = pd.Series({"rows": len(df),
                    "missing_vibration": df["vibration"].isna().sum(),
                    "duplicate_keys": df.duplicated(["machine", "cycle"]).sum(),
                    "outside_range": df["vibration"].gt(10).sum()})
print(report)
print("missing rate", df["vibration"].isna().mean())

**같은 결과를 얻는 방법과 선택 이유**

- 건수는 작업량, 비율은 다른 기간·설비와의 비교에 좋다. 둘 다 보고한다.
- `>10` 같은 범위는 장비 사양에 근거해야 한다. 통계적 이상치와 물리적으로 불가능한 값은 별도로 기록한다.

## Z-score 두 방법

**코드 → 코드 개념**: Z-score는 평균에서의 거리를 표준편차 단위로 나타낸다.

**코드 사용법**: NumPy 수식과 Pandas 수식으로 같은 값을 계산한다.

In [ ]:
x = pd.Series([2.0, 2.2, 2.4, 2.6, 2.8])
z_numpy = (x.to_numpy() - x.mean()) / x.std(ddof=0)
z_pandas = (x - x.mean()) / x.std(ddof=0)
print(z_numpy.round(2), z_pandas.round(2).tolist())
assert np.allclose(z_numpy, z_pandas)

**같은 결과를 얻는 방법과 선택 이유**

- NumPy 배열은 수치 연산만 필요한 경우, Pandas Series는 행 인덱스를 유지하며 원본에 붙일 때 편하다.
- `ddof`를 같게 지정해야 결과가 같다. 표본 표준편차와 모집단 표준편차의 차이도 기록한다.

## 설비별 기준과 강건한 점수

**코드 → 코드 개념**: 설비마다 정상 수준이 다르면 전체 평균으로 표준화하면 오탐이 생긴다.

**코드 사용법**: 그룹별 Z-score와 중앙값 기반 편차를 비교한다.

In [ ]:
sample = pd.DataFrame({"machine": ["A"] * 4 + ["B"] * 4,
                        "vibration": [2.0, 2.1, 2.2, 3.5, 8.0, 8.1, 8.2, 9.5]})
group = sample.groupby("machine")["vibration"]
sample["z"] = group.transform(lambda s: (s - s.mean()) / s.std(ddof=0))
sample["from_median"] = group.transform(lambda s: s - s.median())
print(sample)

**같은 결과를 얻는 방법과 선택 이유**

- 설비별 Z-score는 각 설비의 상대적 변화를 볼 때 좋다. 물리적으로 동일한 절대 한계를 적용해야 한다면 원래 단위의 기준도 유지한다.
- 중앙값 기반 편차는 극단값 영향이 작지만 단위가 원래 값 그대로다. 정상 기준 기간을 먼저 고르는 것이 핵심이다.

## 혼동행렬과 품질 지표

**코드 → 코드 개념**: 탐지 결과는 맞춘 수뿐 아니라 놓친 고장과 오경보를 나눠 평가한다.

**코드 사용법**: TP, FP, FN, TN과 정밀도·재현율을 직접 계산한다.

In [ ]:
actual = np.array([0, 0, 1, 1, 1, 0])
pred = np.array([0, 1, 1, 0, 1, 0])
tp = ((actual == 1) & (pred == 1)).sum()
fp = ((actual == 0) & (pred == 1)).sum()
fn = ((actual == 1) & (pred == 0)).sum()
tn = ((actual == 0) & (pred == 0)).sum()
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print(tp, fp, fn, tn, precision, recall)

**같은 결과를 얻는 방법과 선택 이유**

- 정밀도는 경보 중 실제 고장의 비율, 재현율은 실제 고장 중 잡은 비율이다. 고장을 놓치는 비용이 크면 재현율을 중요하게 본다.
- 정상 데이터가 많으면 정확도만 높아도 탐지가 쓸모없을 수 있다. 실제 운영에서는 경보당 점검 부담도 함께 본다.

## 원본 학습 자료

[`2. practice/06_Z-score`](../2.%20practice/06_Z-score), [`2. practice/07_domain_Predictive_Maintenance`](../2.%20practice/07_domain_Predictive_Maintenance), [`1. lecture/03_domain/steel_study_notes_final/code`](../1.%20lecture/03_domain/steel_study_notes_final/code)